# Lab 5b — Difference-in-Differences: Did the Regional Reminder Rollout Work?
# المعمل 5ب — الفروق في الفروق: هل نجح تفعيل التذكيرات على مستوى المناطق؟

**Module 5 — Quasi-Experiments: Matching, Difference-in-Differences and Instrumental Variables**
**الوحدة 5 — التجارب شبه المضبوطة: المطابقة والفروق في الفروق والمتغيرات الأداتية**

*SDA-DSC-213 · SDAIA Academy · Day 3, Hour 4 · 50 minutes*

---

## The situation

Injaz switched on an SMS/push **reminder programme** region by region over six months —
in four waves, for operational reasons, not experimental ones. Four regions never got it.

The dashboard says completion in the activated regions rose **+4.8 pp** from before to after.
A SAR 90M annual extension hinges on that number.

But national digital adoption was climbing anyway. Some of that +4.8 pp is the programme and
some of it is the tide. **Your job is to separate them.**

The staggered rollout is a gift: at any moment there are regions that have the programme and
regions that do not *yet*. The not-yet-treated regions tell you what the tide alone was doing.

| | |
|---|---|
| **Objective** | Estimate the reminder effect with TWFE DiD + event study, test parallel trends, run a placebo-timing test, and see what a parallel-trends **failure** looks like |
| **Data** | `injaz_regions_panel.csv` (13 regions × 24 months) and `injaz_regions_panel_pretrend.csv` |
| **Estimand** | **ATT** on completion rate in activated regions |
| **Ground truth** | The full effect was **planted** at **+3.0 pp with a two-month ramp-in**. Do not look yet. |

### Bilingual key terms (first use)

| English | العربية |
|---|---|
| Difference-in-differences | الفروق في الفروق |
| Parallel trends | الاتجاهات المتوازية |
| Event study | دراسة الأحداث |
| Two-way fixed effects (TWFE) | التأثيرات الثابتة ثنائية الاتجاه |
| Placebo test | اختبار وهمي |


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
# --- lab scaffolding: self-checks that never crash the notebook ----------
def check(name, fn):
    """Run a self-check. Prints PASS/FAIL/TODO instead of raising, so the
    notebook always runs top-to-bottom even with unfinished exercises."""
    try:
        fn()
    except AssertionError as e:
        print(f"[ TODO ]  {name}: {e}")
        return False
    except Exception as e:
        print(f"[ ERROR ] {name}: {type(e).__name__}: {e}")
        return False
    print(f"[  OK  ]  {name}")
    return True


def needs(*objs, msg="complete the TODO cell above first"):
    """Guard downstream cells so an unfinished TODO skips rather than crashes."""
    if any(o is None for o in objs):
        print(f"[ SKIPPED ] {msg}")
        return False
    return True

print("Scaffolding loaded: use check(...) to self-test and needs(...) to guard.")

## Step 1 — Inspect the panel and the rollout waves *(10 min)*

A DiD analysis is only as good as the panel it sits on. Before estimating anything, know:
who is treated, **when**, and who is never treated.

In [ ]:
from causal_utils import NAVY, BLUE, ORANGE, TEAL, SLATE, MUTED

panel = pd.read_csv(DATA / "injaz_regions_panel.csv")
print(f"{len(panel)} rows · {panel['region'].nunique()} regions × "
      f"{panel['period'].nunique()} months")

waves = (panel.groupby("activation_month")["region"].nunique()
         .rename("n_regions").reset_index())
never = panel.loc[panel["activation_month"].isna(), "region"].nunique()
print("\nActivation waves (period at which the programme switched on):")
print(waves.to_string(index=False))
print(f"Never-treated regions: {never}")
panel.head(3)

In [ ]:
# TODO: compute the NAIVE before/after change for the treated regions only,
#       i.e. mean(completion_rate | treated_ever==1 & post==1)
#          - mean(completion_rate | treated_ever==1 & post==0).
#       Store it in `naive_ba`. This is the +6 pp the dashboard is quoting.
naive_ba = None      # <-- replace

def _c():
    assert naive_ba is not None, "naive_ba is still None"
    assert 0.04 < naive_ba < 0.08, f"expected roughly +6 pp, got {100*naive_ba:+.2f} pp"
check("naive before/after reproduced", _c)

if needs(naive_ba):
    print(f"Naive before/after (treated regions only): {100*naive_ba:+.2f} pp")

### Step 1b — The two-lines picture

The DiD intuition in one chart: treated regions and never-treated regions, plotted over
calendar time. If the two lines move in parallel *before* the first activation, the
never-treated line is a credible stand-in for what the treated regions would have done.

In [ ]:
# TODO: plot mean completion_rate by period for (a) treated_ever==1 and
#       (b) treated_ever==0, mark the four activation periods with vertical lines,
#       and annotate the takeaway.
fig, ax = plt.subplots(figsize=(9.5, 4.6))
# TODO: two ax.plot(...) calls
ax.set_title("TODO")
ax.set_xlabel("Period (month index)")
ax.set_ylabel("Completion rate (percentage points)")
plt.show()

## Step 2 — The identifying assumption, in words, first

> **Parallel trends** (*الاتجاهات المتوازية*): absent the reminder programme, completion in
> the activated regions would have moved *in parallel* with completion in the regions that
> had not yet been activated.

This is **not directly testable** — the counterfactual is missing by construction. What you
*can* do, and must, is show that the **pre-period** was parallel, and then say honestly that
you are extrapolating that parallelism forward.

In [ ]:
IDENTIFYING_ASSUMPTION = (
    "Parallel trends: absent the reminder programme, completion rates in activated "
    "regions would have followed the same time path as in not-yet-activated and "
    "never-treated regions."
)
CREDIBLE_THREAT = (
    "A region-specific campaign, staffing change or service launch that coincided with "
    "reminder activation would violate it - the rollout order was operational, and "
    "operational priorities correlate with other investments."
)
print("IDENTIFYING ASSUMPTION\n  " + IDENTIFYING_ASSUMPTION)
print("\nCREDIBLE THREAT\n  " + CREDIBLE_THREAT)
print("\nHOW WE SUPPORT IT\n  1. event study: pre-period coefficients flat and ~zero"
      "\n  2. placebo timing: pretend activation happened 6 months earlier -> null")

## Step 3 — TWFE difference-in-differences *(12 min)*

The regression form:

```
completion_rate ~ treated_post + C(region) + C(period)
```

Region fixed effects absorb every permanent difference between regions. Period fixed effects
absorb the national trend. What is left on `treated_post` is the DiD estimate.

**Standard errors must be clustered on region.** Months within a region are correlated;
pretending otherwise gives an interval that is far too narrow. With only 13 clusters, treat
the interval as approximate and say so.

In [ ]:
# TODO: estimate the TWFE DiD with did_twfe(panel, outcome, unit, time, treated_post).
#       Cluster on region (the default). Store the DiDResult in `did`.
from causal_utils import did_twfe

did = None      # <-- replace

def _c():
    assert did is not None, "did is still None"
    assert 0.015 < did.att < 0.035, f"DiD of {100*did.att:+.2f} pp is outside the plausible band"
    assert did.n_units == 13, "all 13 regions should contribute"
check("TWFE DiD estimated with clustered SEs", _c)

if needs(did):
    print(did)

## Step 4 — The event study: the parallel-trends exhibit *(12 min)*

This is the single most important chart in observational programme evaluation, and the one
most likely to appear in your capstone.

Instead of one average effect, estimate **one coefficient per month relative to activation**,
with month −1 omitted as the baseline. Then read it in this order:

1. **Are the pre-period coefficients flat and centred on zero?** If not, parallel trends is
   already contradicted and the DiD number is not a causal estimate, whatever its p-value.
2. **Does the effect appear at activation, not before it?**
3. **Does it ramp, persist or decay?** That shapes the scaling decision as much as the
   average does.

In [ ]:
# TODO: run event_study(panel, outcome, unit, time, rel_time, window=(-6, 8))
#       with rel_time="months_since_activation". Store the DataFrame in `es`.
from causal_utils import event_study

es = None      # <-- replace

def _c():
    assert es is not None, "es is still None"
    pre = es[(es.period == "pre") & (~es.is_baseline)]
    assert pre["coef"].abs().max() < 0.01, "pre-period coefficients are not flat"
    assert es.loc[es.rel_time == 0, "coef"].iat[0] > 0.005, "no jump at activation"
check("event study: flat pre-period, jump at activation", _c)

if needs(es):
    print(es.round(4).to_string(index=False))

### Step 4b — Plot it

A reusable plotting function — you will call it again in a moment on a *broken* dataset, and
again in the capstone.

In [ ]:
# TODO: write `plot_event_study(es, ax=None, title=..., subtitle=...)`.
#   Requirements:
#     - x = rel_time, y = coef * 100 (percentage points)
#     - vertical CI bars from ci_low*100 to ci_high*100
#     - horizontal line at 0 and a vertical line at rel_time = -0.5 (activation)
#     - pre-period points in SLATE, post-period points in BLUE
#     - title, axis labels in percentage points, and an annotation with the takeaway
def plot_event_study(es_df, ax=None, title="Event study", subtitle=""):
    """TODO: implement. Returns the matplotlib axis."""
    raise NotImplementedError("plot_event_study is not implemented yet")


try:
    if needs(es):
        plot_event_study(es, title="Reminder rollout — event study")
        plt.show()
except NotImplementedError as exc:
    print(f"[ TODO ] {exc}")

## Step 5 — What a parallel-trends **failure** looks like *(10 min)*

Everything above assumed the pre-period was flat, and it was. Now run the identical analysis
on `injaz_regions_panel_pretrend.csv` — the same rollout, except the treated regions were
already pulling away from the controls *before* activation.

**You must be able to recognise this chart instantly.** In the wild nobody labels the file
"pretrend".

In [ ]:
# TODO: load injaz_regions_panel_pretrend.csv into `panel_bad`, run the SAME
#       did_twfe and event_study on it, and store them in `did_bad` / `es_bad`.
panel_bad = None
did_bad = None
es_bad = None

def _c():
    assert es_bad is not None, "es_bad is still None"
    pre = es_bad[(es_bad.period == "pre") & (~es_bad.is_baseline)]
    assert pre["coef"].abs().max() > 0.01, \
        "you should be seeing a NON-flat pre-period here - check you loaded the pretrend file"
check("parallel-trends violation detected in the pretrend panel", _c)

if needs(did_bad):
    print(did_bad)

### Step 5b — Side by side: the credibility exhibit

Same estimator. Same rollout. One of these DiD estimates means something.

In [ ]:
# TODO: draw both event studies side by side with a shared y-axis, using
#       plot_event_study(...) twice on one figure (plt.subplots(1, 2, sharey=True)).
#       Annotate each with its verdict.
print("[ TODO ] side-by-side event-study figure")

## Step 6 — Placebo-timing test *(8 min)*

The event study is a *visual* test. The placebo-timing test is its *numeric* twin: pretend
activation happened **six months earlier than it did**, throw away all genuinely-treated
observations, and re-estimate. Since nothing actually happened at the fake date, the estimate
must be null.

Run it on both panels. It should pass on the clean one and fail on the broken one.

In [ ]:
# TODO: implement `placebo_timing(panel_df, shift=6)`:
#   1. copy the panel; make `fake_activation = activation_month - shift`
#   2. KEEP ONLY genuinely untreated rows: activation_month is NaN, OR period < activation_month
#   3. build `placebo_post` = 1 when treated_ever==1 AND period >= fake_activation
#   4. return the modified frame
#   Then run did_twfe on it with treated_post="placebo_post" for BOTH panels
#   and store the results in `pl_clean` and `pl_bad`.
def placebo_timing(panel_df, shift=6):
    """TODO: implement (see the four steps above)."""
    raise NotImplementedError("placebo_timing is not implemented yet")


pl_clean = None
pl_bad = None

def _c():
    assert pl_clean is not None and pl_bad is not None, "placebo results still None"
    assert pl_clean.p_value > 0.05, "placebo on the CLEAN panel should be null"
    assert pl_bad.p_value < 0.05, "placebo on the PRETREND panel should fire"
check("placebo timing: null on clean, fires on pretrend", _c)

## Step 7 — The staggered-rollout caveat *(6 min)*

Modern DiD literature has a warning for exactly this design: with **staggered** adoption and
effects that change over time, TWFE implicitly uses **already-treated** units as controls,
and some of the implicit comparison weights can be **negative**. In bad cases the estimate is
biased, occasionally even wrong-signed.

The fixes are cohort-aware estimators — **Callaway–Sant'Anna**, **Sun–Abraham**. Neither is
installed here, and neither is needed to make the point. Instead, run the **clean 2×2**: the
first wave (activated at period 10) against the **never-treated** regions only. No
already-treated unit is ever used as a control in that comparison.

If the clean comparison agrees with full TWFE, the negative-weights problem is not biting
here — and you can say that in the memo instead of hand-waving.

In [ ]:
# TODO: build `clean_2x2`: rows where activation_month == 10 OR activation_month is NaN.
#       Run did_twfe on it (treated_post) and store the result in `did_clean`.
#       Compare to the full-panel TWFE estimate.
clean_2x2 = None
did_clean = None

def _c():
    assert did_clean is not None, "did_clean is still None"
    assert did_clean.n_units == 6, "wave-1 regions + never-treated regions should be 6 units"
    assert abs(did_clean.att - did.att) < 0.01, \
        "a >1 pp gap here would be evidence of staggered-TWFE bias"
check("clean not-yet-treated comparison agrees with full TWFE", _c)

if needs(did_clean):
    print(did_clean)

### Optional — the DoWhy / cohort-estimator equivalents

Not installed here. The cell degrades gracefully and explains the mapping.

In [ ]:
# --- OPTIONAL: packages you would reach for outside this training image ----
try:
    from dowhy import CausalModel                       # noqa: F401
    print("DoWhy available - a DiD estimand can be declared via a graph with "
          "unit and time fixed effects.")
except ImportError:
    print("DoWhy not installed - expected. The identify -> estimate -> refute\n"
          "workflow was still executed in full above:\n"
          "  identify -> parallel-trends assumption stated in Step 2\n"
          "  estimate -> did_twfe (Step 3) + event_study (Step 4)\n"
          "  refute   -> pre-trend inspection (Step 5) + placebo timing (Step 6)\n"
          "              + clean not-yet-treated comparison (Step 7)")

try:
    import differences                                   # noqa: F401
    print("Callaway-Sant'Anna available via the `differences` package.")
except ImportError:
    print("\nNo staggered-DiD package (Callaway-Sant'Anna / Sun-Abraham) installed.\n"
          "Step 7's clean wave-1-vs-never-treated 2x2 is the assumption-light\n"
          "substitute and is sufficient for the capstone. Implementing a cohort-aware\n"
          "estimator and comparing it against naive TWFE is a listed capstone extension.")

## Step 8 — Compare against the planted ground truth

In [ ]:
import json
GT = json.loads((DATA / "GROUND_TRUTH.json").read_text())["module5_regions_panel"]


def ground_truth_comparison():
    settled = float(es[es["rel_time"] >= 2]["coef"].mean())
    rows = [
        {"quantity": "Naive before/after (treated only)", "pp": 100*naive_ba,
         "note": "contaminated by the national trend"},
        {"quantity": "TWFE DiD (all 13 regions)", "pp": 100*did.att,
         "note": f"[{100*did.ci_low:+.2f}, {100*did.ci_high:+.2f}] clustered"},
        {"quantity": "Clean wave-1 vs never-treated", "pp": 100*did_clean.att,
         "note": "no already-treated unit used as a control"},
        {"quantity": "Event study, settled effect (months 2+)", "pp": 100*settled,
         "note": "the MATURE effect, after ramp-in"},
        {"quantity": "PLANTED full ATT", "pp": 100*GT["true_ATT"],
         "note": "known by construction, with a 2-month ramp-in"},
        {"quantity": "Pretrend panel TWFE (INVALID)", "pp": 100*did_bad.att,
         "note": "parallel trends fails - do not report this number"},
    ]
    out = pd.DataFrame(rows)
    print(out.to_string(index=False, float_format=lambda v: f"{v:+.2f}"))

    print(f"\n  planted full effect          : {100*GT['true_ATT']:+.2f} pp")
    print(f"  TWFE recovered               : {100*did.att:+.2f} pp   "
          f"(gap {100*(did.att - GT['true_ATT']):+.2f} pp)")
    print(f"  event-study settled effect   : {100*settled:+.2f} pp   "
          f"(gap {100*(settled - GT['true_ATT']):+.2f} pp)")
    print("""
WHY TWFE LANDS BELOW THE PLANTED +3.0 pp - AND WHY THAT IS NOT A BUG
--------------------------------------------------------------------
The programme was built with a TWO-MONTH RAMP-IN: only ~60% of the effect is
present in the first two months after activation. TWFE reports the AVERAGE
effect across every post-activation month it observes, ramp-in months included,
so it necessarily lands below the mature effect. The event study separates them:
months 0-1 sit near +1.7 to +1.9 pp, months 2+ settle at +3.0 pp.

Both numbers are correct answers to different questions:
  "what did the programme deliver on average last year?"  -> +2.5 pp  (TWFE)
  "what will it deliver in a mature region next year?"    -> +3.0 pp  (event study)
A scaling decision needs the SECOND number. A retrospective cost-benefit of
last year needs the FIRST. Report both, and say which is which.""")
    return out


if needs(naive_ba, did, did_clean, es, did_bad):
    comparison = ground_truth_comparison()

    def _c():
        assert abs(did.att - GT["twfe_estimate_clean"]) < 0.002, "TWFE does not match the answer key"
        settled = float(es[es["rel_time"] >= 2]["coef"].mean())
        assert abs(settled - GT["true_ATT"]) < 0.005, \
            "the settled event-study effect should recover the planted +3.0 pp"
    check("TWFE matches the key; settled effect recovers the planted +3.0 pp", _c)

## Step 9 — Write the finding

In [ ]:
def render_finding():
    settled = float(es[es["rel_time"] >= 2]["coef"].mean())
    return f"""
FINDING - Regional reminder programme
-------------------------------------
The reminder programme raised service completion in activated regions by
{100*did.att:+.2f} percentage points on average across the observed post-period
(95% CI {100*did.ci_low:+.2f} to {100*did.ci_high:+.2f}, standard errors clustered
on 13 regions). The dashboard's +{100*naive_ba:.1f} pp before/after figure
overstates it by {100*(naive_ba - did.att):.1f} points because national digital
adoption was rising in every region, treated or not.

The effect RAMPS: months 0-1 after activation show roughly
{100*es[es.rel_time.isin([0, 1])]['coef'].mean():+.1f} pp and months 2 onward settle at
{100*settled:+.1f} pp. For a SCALING decision, {100*settled:+.1f} pp is the relevant
figure; for a retrospective valuation of last year, {100*did.att:+.2f} pp is.

ESTIMAND   ATT on completion rate in activated regions.
ASSUMPTION {IDENTIFYING_ASSUMPTION}
EVIDENCE   Event study: all six pre-period coefficients within
           {100*es[(es.period == 'pre') & (~es.is_baseline)]['coef'].abs().max():.2f} pp of
           zero with CIs covering zero. Placebo timing (activation shifted 6
           months early): {100*pl_clean.att:+.2f} pp, p = {pl_clean.p_value:.2f} - null.
           Clean wave-1-vs-never-treated 2x2: {100*did_clean.att:+.2f} pp, so
           staggered-TWFE negative weights are not materially biting.
THREAT     {CREDIBLE_THREAT}
CAVEAT     Only 13 clusters. Cluster-robust intervals are approximate at this
           cluster count and should be read as indicative, not exact.
DECISION   SCALE. The evidence is the strongest of the three Injaz programmes:
           the assumption is supported by a testable pre-period, not merely
           asserted.
"""


if needs(did, naive_ba, es, pl_clean, did_clean):
    FINDING = render_finding()
    print(FINDING)

## What you now own

A reusable **DiD evaluation kit** — and the event-study plotting function, which is the
single most portable thing you will build all week.

In [ ]:
def build_artefact():
    return {
        "notebook": "lab5b - difference-in-differences + event study",
        "reusable_pipeline": [
            "1. inspect the panel: waves, never-treated units, the two-lines chart",
            "2. reproduce the naive before/after (the number you must argue down)",
            "3. WRITE the parallel-trends assumption and its most credible threat",
            "4. did_twfe(...) with unit-clustered SEs -> the headline ATT",
            "5. event_study(...) + plot_event_study(...) -> the credibility exhibit",
            "6. run the SAME event study on a pretrend panel so you can recognise failure",
            "7. placebo_timing(...) -> numeric falsification test",
            "8. clean not-yet-treated 2x2 -> bound the staggered-TWFE risk",
            "9. FINDING: average effect AND settled effect, assumption, evidence, threat",
        ],
        "reusable_functions_written_here": ["plot_event_study", "placebo_timing"],
        "artefacts_for_the_capstone": ["event-study plot", "side-by-side pass/fail figure",
                                       "placebo-timing line", "FINDING paragraph"],
        "numbers_recovered": {
            "naive_before_after_pp": round(100*naive_ba, 2),
            "twfe_att_pp": round(100*did.att, 2),
            "settled_effect_pp": round(100*es[es["rel_time"] >= 2]["coef"].mean(), 2),
            "clean_2x2_pp": round(100*did_clean.att, 2),
            "pretrend_panel_att_pp": round(100*did_bad.att, 2),
            "placebo_timing_p": round(pl_clean.p_value, 3),
        },
        "rule_you_can_now_state": (
            "A DiD number without an event study is an assertion. The pre-period is the "
            "only part of parallel trends you can test, so test it, show it, and say "
            "plainly that the post-period parallelism remains an assumption."
        ),
    }


if needs(did, naive_ba, es, did_clean, did_bad, pl_clean):
    ARTEFACT = build_artefact()
    for k, v in ARTEFACT.items():
        print(f"\n{k}:")
        print("  " + (("\n  ".join(v)) if isinstance(v, list) else str(v)))

## Mini exercises

### 1. Debugging — the post-treatment control *(6 min)*

A colleague's DiD notebook "controls for volume" by adding `reminders_sent` to the
regression, reasoning that regions sending more reminders should see more completions.

Run it. Then explain what happened and why.

In [ ]:
# TODO: fit  completion_rate ~ treated_post + reminders_sent + C(region) + C(period)
#       with region-clustered SEs, read off the treated_post coefficient, and write
#       your diagnosis in DIAGNOSIS.
import statsmodels.formula.api as smf

bad_did = None     # <-- replace with the treated_post coefficient (a float)
DIAGNOSIS = ""     # <-- replace

if needs(bad_did):
    print(f"correct DiD                     : {100*did.att:+.2f} pp")
    print(f"with reminders_sent 'controlled': {100*bad_did:+.2f} pp")
    print(DIAGNOSIS)

### 2. Case-study discussion — "Did the Regional Reminder Programme Work?" *(6 min)*

Answer in the strings below. These are the four discussion questions from the module case
study; your answers go straight into your capstone memo.

1. Why is the naive before/after +4.8 pp an overestimate here?
2. How does using not-yet-treated regions as controls protect the estimate?
3. What would a non-flat pre-trend tell you — and would you still report a number?
4. How do you communicate "+2.6 pp, assuming parallel trends" to a decision-maker honestly?

In [ ]:
CASE_STUDY = {
    "q1_why_naive_overstates": "TODO",
    "q2_why_not_yet_treated_controls": "TODO",
    "q3_non_flat_pretrend": "TODO",
    "q4_communicating_the_assumption": "TODO",
}
for k, v in CASE_STUDY.items():
    print(f"\n{k}:\n  {v}")

### 3. Assumption courtroom *(5 min, in pairs)*

Pair A defends the +2.6 pp estimate. Pair B cross-examines the parallel-trends assumption.
The class rules **credible** or **not credible**.

Pair B's strongest lines of attack, in order of force:

1. *"The rollout order was chosen by the operations team. What did they optimise for?"* — if
   they activated the regions with the most headroom first, trends were never parallel.
2. *"Thirteen clusters. Your standard errors are fiction."*
3. *"Your event study omits month −1 as the baseline. What if the programme leaked into
   month −1 through pre-launch communications?"*

Pair A's honest defences: the flat pre-period across six months; the null placebo; the clean
wave-1-vs-never-treated 2×2; and — for attack 3 — re-running the event study with month −3
as the baseline.

---

### Quiz recall (Module 5, questions 2 and 14)

2. **State the DiD identifying assumption.** → parallel trends: absent treatment, the groups
   would have moved in parallel.
14. **How is it supported?** → flat pre-period coefficients in an event study (plus a null
    placebo-timing test).

### Next

**Lab 6** — where the *rules* come from. Both Lab 5a's adjustment set and Lab 5b's "never
control for a post-treatment variable" are consequences of one criterion, and you will derive
it from a graph.